# Realised weather features
ERA5 reanalysis is delayed six calendar days as a conservative five-day-release-plus-timing buffer. This remains revised reanalysis, not a vintage observation archive. Fixed 2023 demand weights are inherited from the source. Normals use prior years only.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("TTF_PROJECT_ROOT", Path.cwd().parent))
RAW = ROOT / "1. Raw Data"
OUT = ROOT / "2. Data Processing"
wx = pd.read_csv(RAW / "weather_hist.csv", parse_dates=["date"])
wx = wx.dropna(subset=["EU_HDD", "EU_CDD"]).sort_values("date").reset_index(drop=True)
wx = wx.rename(columns={"EU_HDD": "hdd", "EU_CDD": "cdd"})

In [2]:
feb29 = wx["date"].dt.month.eq(2) & wx["date"].dt.day.eq(29)
key = wx["date"].dt.strftime("%m-%d").mask(feb29, "02-28")
year = wx["date"].dt.year
hist = wx.loc[~feb29, ["date", "hdd", "cdd"]].copy()
hist["key"] = hist["date"].dt.strftime("%m-%d")
hist["year"] = hist["date"].dt.year
for col, out in [("hdd", "hdd_anom7"), ("cdd", "cdd_anom7")]:
    lookup = hist.set_index(["key", "year"])[col]
    prior = pd.DataFrame({lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup) for lag in range(1, 11)})
    anom = wx[col] - prior.mean(axis=1, skipna=False)
    wx[out] = anom.rolling(7, min_periods=7).sum()
wx = wx.drop(columns=["EU_TEMP"])
wx["wx_source_date"] = wx["date"]
wx["wx_available_date"] = wx["wx_source_date"] + pd.Timedelta(days=6)
wx["weather_revised_proxy"] = True
wx["date"] = wx["wx_available_date"]
wx.to_csv(OUT / "weather_hist_processed.csv", index=False)